
# Quasi-RAW

Simple example.

Example on how to run the brain parcellation pre-processing using BrainPrep.
See `user guide <quasiraw>` for details.

## Data

Let's first get some anatomical data.


In [ ]:
from pathlib import Path
from brainprep.utils import Bunch
from brainprep.datasets import OpenMSDataset

datadir = Path("/tmp/brainprep-data")
datadir.mkdir(parents=True, exist_ok=True)
dataset = OpenMSDataset(datadir)
data = Bunch(
    sub01=dataset.fetch(
        subject="01",
        modality="T1w",
        dtype="cross_sectional",
    ),
    sub02=dataset.fetch(
        subject="02",
        modality="T1w",
        dtype="cross_sectional",
    ),
)
print(data)

## Analysis

Let's now perform the preprocessing using BrainPrep.
As with many tutorials, we won't execute the code directly here.
However, feel free to set the 'dryrun' configuration to False
to actually run each step and generate results on disk.



In [ ]:
from brainprep.workflow import (
    brainprep_quasiraw,
    brainprep_group_quasiraw,
)
from brainprep.config import Config
from brainprep.reporting import RSTReport

outdir = Path("/tmp/brainprep-quasiraw")
outdir.mkdir(parents=True, exist_ok=True)
with Config(dryrun=True, verbose=True):
    for subject_data in data.values():
        report = RSTReport()
        brainprep_quasiraw(
            anatomical_file=subject_data.anat,
            output_dir=outdir,
            keep_intermediate=True,
        )
        print(report)
    outputs = brainprep_group_quasiraw(
        output_dir=outdir,
    )

## CLI

Let's now generate the same analysis using the CLI. The goal here is to
translate the workflow calls into explicit shell commands.
See `user guide <cli>` for details.



In [ ]:
from pprint import pprint

commands = []
commands.append(
    [
        [
            "brainprep", "subject-level-quasiraw",
            "--anatomical_file", str(subject_data.anat),
            "--output-dir", str(outdir),
            "--keep-intermediate",
        ] for subject_data in data.values()
    ]
)
commands.append(
    [
        [
            "brainprep", "group-level-quasiraw",
            "--output-dir", str(outdir),
        ]
    ]
)
pprint(commands)

## Container

Note that the commands generated by the CLI are not limited to being
displayed for reference; they can also be executed directly within the
workflow‑dedicated container. By running the commands inside the container,
you benefit from a controlled runtime context where all necessary
dependencies, libraries, and configuration files are already available.
In practice, this means that once the CLI has produced the appropriate
instructions, you can simply copy and run them inside the container to
achieve the intended results. You can find the BrainPrep images on Docker
Hub: [Neurospin Docker Hub](https://hub.docker.com/u/neurospin).

